# SelfQueryRetriever
- 사용자의 자연어 질문을 분석하여 벡터 검색에 사용할 검색어와 metadata filter 조건을 함께 생성하는 retriever이다.

In [1]:
%pip install -U langchain-classic lark

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 환결 설정

In [1]:
from dotenv import load_dotenv
load_dotenv()

# PINECONE_INDEX_NAME = 'adv-rag'
PINECONE_META_INDEX_NAME = 'adv-meta-rag'
PINECONE_INDEX_REGION = 'us-east-1'
PINECONE_INDEX_CLOUD = 'aws'
PINECONE_INDEX_METRIc = 'cosine'
PINECONE_INDEX_DEMENSION = 1536

OPENAI_LLM_MODEL = 'gpt-4.1-mini'
OPENAI_EMBEDDING_MODEL = 'text-embedding-3-small'

## 데이터 로드

In [2]:
import pandas as pd

document_df = pd.read_csv("data/documents_meta.csv")
queries_df = pd.read_csv("data/queries_meta.csv")

document_df.head()

,doc_id,title,content,author,category
0,D1,제주도 여행 가이드,"제주도는 대한민국의 대표 관광지로서, 한라산 등반, 성산 일출봉 관광, 해변 활동(...",박민준,여행;제주;관광
1,D2,전주 비빔밥과 진주 비빔밥 차이점,"비빔밥은 조선 시대부터 전해 내려온 대표적 한국 음식으로, 밥 위에 고명(채소·고기...",이서연,음식;비빔밥;역사
2,D3,걸스데이 히트곡 분석,"걸스데이는 2010년 데뷔한 대한민국의 4인조 걸그룹으로, 대표곡으로는 “Somet...",최유나,연예;음악;대중문화
3,D4,세종대왕과 훈민정음,세종대왕(1397~1450)은 훈민정음을 창제하여 한글을 보급한 조선의 4대 임금입...,정하늘,역사;문화;문자
4,D5,이순신 장군의 명량 해전,이순신 장군(1545~1598)은 임진왜란 당시 명량 해전에서 13척의 배로 133...,정하늘,역사;군사;전쟁


## 벡터 스토어 준비

In [3]:
from langchain_openai import OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore

# 임베딩 모델 생성
embeddings = OpenAIEmbeddings(model=OPENAI_EMBEDDING_MODEL)

# 벡터 스토어 생성
vector_store = PineconeVectorStore(
    index_name=PINECONE_META_INDEX_NAME,
    embedding=embeddings
)

## 출력 함수

In [4]:
def print_docs(docs):
    for doc in docs:
        print(f"{doc.metadata['doc_id']} : ")
        print('author : ',doc.metadata.get('author'))
        print('category : ',doc.metadata.get('category'))
        print(doc.page_content)
        print()

def doc_ids(docs):
    return [doc.metadata['doc_id'] for doc in docs]

## SelfQueryRetriever 생성

In [20]:
from langchain_openai import ChatOpenAI
from langchain_classic.retrievers.self_query.base import SelfQueryRetriever
from langchain_classic.chains.query_constructor.schema import AttributeInfo

llm = ChatOpenAI(model=OPENAI_LLM_MODEL, temperature=0)

# SelfQueryRetriever가 자연어 질문에서 추출할 수 있는 metadata 필드 정보
metadata_field_info = [
    AttributeInfo(
        name='author',
        type='string',
        description=(
            '문서를 작성한 저자 이름. '
            '예: 김철수, 한지민, 이서연, 오세린, 박민준, 김도윤, 박준호, 최유나, 강민재, 정하늘'
        )
    ),
    AttributeInfo(
        name='category',
        type='list[string]',
        description=(
            '문서가 속한 카테고리 이름 목록. '
            '예: 검색, RAG, 벡터DB, AI, 기술, 역사, 음식, 여행, 디자인패턴, 프로그래밍, 문화, 환경, 스포츠, 게임, 영화'
        )
    ),
]

# SelfQueryRetriever 객체 생성
self_query_retriever = SelfQueryRetriever.from_llm(
    llm=llm,
    vectorstore=vector_store,
    document_contents='여행, 음식, 역사, AI, 검색 기술, RAG, 디자인패턴, 프로그래밍 등에 대한 설명 문서',
    metadata_field_info=metadata_field_info,
    search_kwargs={'k': 5},
)

## 자연어 질문이 구조화 되는 과정 확인

In [10]:
query_conductor = self_query_retriever.query_constructor

def inspect_self_query(user_query):
    print('원본 질문')
    print(user_query)
    print()

    structured_query = query_conductor.invoke({'query':user_query})

    print('구조화된 검색 조건')
    print(structured_query)
    print()

    docs = self_query_retriever.invoke(user_query)
    
    print('검색 결과 ID')
    print(doc_ids(docs))
    print()

    return structured_query, docs

## 단일 질문으로 흐름 확인

In [11]:
user_query = '김철수가 작성한 검색 카테고리 문서중 ChromaDB와 Qdrant를 비교한 문서를 찾아줘'

structured_query, results = inspect_self_query(user_query)
print_docs(results)

원본 질문
김철수가 작성한 검색 카테고리 문서중 ChromaDB와 Qdrant를 비교한 문서를 찾아줘

구조화된 검색 조건
query='ChromaDB와 Qdrant 비교' filter=Operation(operator=<Operator.AND: 'and'>, arguments=[Comparison(comparator=<Comparator.EQ: 'eq'>, attribute='author', value='김철수'), Comparison(comparator=<Comparator.IN: 'in'>, attribute='category', value='검색')]) limit=None

검색 결과 ID
['D27', 'D29', 'D28', 'D30']

D27 : 
author :  김철수
category :  ['검색', '벡터DB', '기술']
ChromaDB와 Qdrant는 벡터 검색 라이브러리로, ChromaDB는 오픈소스 벡터 DB로 간단한 파이썬 인터페이스를 제공하며, Qdrant는 Rust 기반 고성능 벡터 DB로 GPU 가속 지원 및 필터링 기능이 강점입니다. 성능 비교 실험 시 인덱싱 속도, 검색 응답 속도, 메모리 사용량, 스케일링 용이성 등을 비교합니다. 파이썬 코드 예제와 벤치마크 결과가 공개되어 있어, 개발자가 선택하기 용이합니다.

D29 : 
author :  김철수
category :  ['검색', 'RAG', '메타데이터']
Self-Query Retriever는 문서 메타데이터(제목·요약·키워드)를 분석해, 사용자가 실제로 검색할 만한 쿼리를 GPT-4o-mini 등 생성형 모델로 생성한 뒤, 생성된 가상 쿼리를 다시 검색에 활용하는 기법입니다. 이 과정을 통해 사용자가 입력한 실제 질의보다 검색 품질을 높이는 효과를 얻을 수 있으며, 생성된 쿼리는 ‘Self-Query’라고 불립니다.

D28 : 
author :  김철수
category :  ['검색', 'RAG', '압축']
Contextual Compression은 긴 텍스

## 난이도별 질문 확인

In [12]:
text_queries = [
    '김철수가 작성한 문서를 모두 찾아줘',
    '카테고리가 검색인 문서들을 찾아줘',
    '박준호 저자의 디자인 패턴 카테고리 문서 중 싱글톤 패턴과 멀티스레드 안정성을 설명한 자료는?'
]

for query in text_queries:
    structured_query, results = inspect_self_query(user_query)

원본 질문
김철수가 작성한 검색 카테고리 문서중 ChromaDB와 Qdrant를 비교한 문서를 찾아줘

구조화된 검색 조건
query='ChromaDB Qdrant 비교' filter=Operation(operator=<Operator.AND: 'and'>, arguments=[Comparison(comparator=<Comparator.EQ: 'eq'>, attribute='author', value='김철수'), Comparison(comparator=<Comparator.IN: 'in'>, attribute='category', value='검색')]) limit=None

검색 결과 ID
['D27', 'D29', 'D28', 'D30']

원본 질문
김철수가 작성한 검색 카테고리 문서중 ChromaDB와 Qdrant를 비교한 문서를 찾아줘

구조화된 검색 조건
query='ChromaDB와 Qdrant 비교' filter=Operation(operator=<Operator.AND: 'and'>, arguments=[Comparison(comparator=<Comparator.EQ: 'eq'>, attribute='author', value='김철수'), Comparison(comparator=<Comparator.IN: 'in'>, attribute='category', value='검색')]) limit=None

검색 결과 ID
['D27', 'D29', 'D28', 'D30']

원본 질문
김철수가 작성한 검색 카테고리 문서중 ChromaDB와 Qdrant를 비교한 문서를 찾아줘

구조화된 검색 조건
query='ChromaDB와 Qdrant 비교' filter=Operation(operator=<Operator.AND: 'and'>, arguments=[Comparison(comparator=<Comparator.EQ: 'eq'>, attribute='author', value='김철수'), Comparison(comparator=<Compar

## 전체 질의 실행
- queries_meta.csv으 전체 질문을 실행한다.

In [21]:
from tqdm import tqdm

self_query_results = {}

for idx,row in tqdm(queries_df.iterrows(), total=len(queries_df)):
    qid = row['query_id']
    query_text = row['query_text']

    docs = self_query_retriever.invoke(query_text)

    self_query_results[qid] = doc_ids(docs)

100%|██████████| 30/30 [01:00<00:00,  2.00s/it]


## 평가 함수 준비

In [22]:
import numpy as np

def parse_relevant(relevant_str):
    """다중 정답 및 등급을 처리하기 위한 헬퍼 함수"""
    pairs = relevant_str.split(";")
    rel_dict = {}
    for pair in pairs:
        doc_id, grade = pair.split("=")
        rel_dict[doc_id] = grade
    return rel_dict 

def compute_metrics(predicted, relevant_dict, k=5):
    relevant_docs = set(relevant_dict.keys())
    top_k = predicted[:k]
    hits = sum(1 for doc in top_k if doc in relevant_docs)
    precision = hits / k
    total_relevant = len(relevant_docs)
    recall = hits / total_relevant if total_relevant > 0 else 0 
    rr = 0
    for idx, doc in enumerate(top_k):
        if doc in relevant_docs:
            rr = 1 / (idx + 1)
            break
    num_correct = 0
    precision_sum = 0
    for i, doc in enumerate(top_k):
        if doc in relevant_docs:
            num_correct += 1
            precision_sum += num_correct / (i + 1)
    denominator = min(total_relevant, k)
    ap = precision_sum / denominator if denominator > 0 else 0
    return precision, recall, rr, ap

def evaluate_all(method_results, queries_df, k=5):
    prec_list, rec_list, rr_list, ap_list = [], [], [], []
    for idx, row in queries_df.iterrows():
        qid = row['query_id']
        relevant_dict = parse_relevant(row['relevant_doc_ids'])
        predicted = method_results[qid]
        p, r, rr, ap = compute_metrics(predicted, relevant_dict, k)
        prec_list.append(p)
        rec_list.append(r)
        rr_list.append(rr)
        ap_list.append(ap)
    return {
        'Precision@k' : np.mean(prec_list),
        'Recall@k' : np.mean(rec_list),
        'MRR' : np.mean(rr_list),
        'MAP' : np.mean(ap_list),
    }

## dense 검색과 selfqueryretriever 검색 비교

In [23]:
dense_results = {}

for idx, row in tqdm(queries_df.iterrows(),total=len(queries_df)):
    qid = row['query_id']
    query_text = row['query_text']

    dense_docs = vector_store.similarity_search(query_text,k=5)
    dense_results[qid] = doc_ids(dense_docs)

100%|██████████| 30/30 [00:13<00:00,  2.22it/s]


In [24]:
dense_metrics = evaluate_all(dense_results,queries_df)
self_query_metrics = evaluate_all(self_query_results,queries_df)

metrics_df = pd.DataFrame({
    'Metric' : ["Precision@k","Recall@k","MRR","MAP"],
    'Dense' : [dense_metrics["Precision@k"],dense_metrics["Recall@k"],dense_metrics["MRR"],dense_metrics["MAP"]],
    'selfqueryretriever' : [self_query_metrics["Precision@k"],self_query_metrics["Recall@k"],self_query_metrics["MRR"],self_query_metrics["MAP"]],
})
metrics_df

,Metric,Dense,selfqueryretriever
0,Precision@k,0.253333,0.286667
1,Recall@k,0.941667,0.966667
2,MRR,0.941667,0.966667
3,MAP,0.887778,0.961111
